In [70]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from xgboost import XGBClassifier
import os
import pandas as pd
from glob import glob
import rasterio as rio
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import NearestNeighbors
from imblearn.over_sampling import SMOTE

from src.mslandcover.models import HRNetSegmentationModel
from src.mslandcover.config import HRNET_W18_CONFIG

In [48]:
target_paths = glob('data/png_images/batch_0/target/*.tif')
input_paths = [x.replace('target', 'input_tif') for x in target_paths]

img_paths_df = pd.DataFrame({'input': input_paths, 'target': target_paths})
img_paths_df['X'] = img_paths_df['input'].apply(lambda x: rio.open(x).read())
img_paths_df['y'] = img_paths_df['target'].apply(lambda x: rio.open(x).read())

dataset_mean = torch.load('./weights/pretrain_mean.pth', weights_only=True).numpy()
dataset_std = torch.load('./weights/pretrain_std.pth', weights_only=True).numpy()

img_paths_df['X_norm'] = img_paths_df['X'].apply(lambda x: ((x.transpose(1, 2, 0) - dataset_mean) / dataset_std).transpose(2, 0, 1))

In [49]:
print(len(img_paths_df))

90


In [57]:
for weights in ['imagenet', 'hsv_simclr']:
    model = HRNetSegmentationModel(HRNET_W18_CONFIG, img_decoder_head=False, aux_simclr_head=False)
    model = model.to('cuda')
    model.eval()
    
    z_list = []
    if weights == 'imagenet':
        model.load_encoder_weights(torch.load('weights/hrnet_w18/imagenet.pth', weights_only=True))
    else:
        model.load_state_dict(torch.load('weights/hrnet_w18/hsv_simclr_old.pth', weights_only=True), strict=False)
    
    for i in range(0, len(img_paths_df), 16):
        print(i)
        X_batch = torch.from_numpy(np.stack(img_paths_df['X_norm'].values[i:i+16])).to('cuda').to(torch.float32)
        # interpolate to 256x256
        z_batch = model.encoder(X_batch)
        z_batch = F.interpolate(z_batch, size=(256, 256), mode='bilinear', align_corners=True)
        z_batch = z_batch.detach().to('cpu').numpy()
        z_list.extend(list(z_batch))
    
    img_paths_df[f'z_{weights}'] = z_list
    
n_deep_features = img_paths_df['z_imagenet'].values[0].shape[0]
print(f'Number of deep features per pixel: {n_deep_features}')

0
16
32
48
64
80
0
16
32
48
64
80
Number of deep features per pixel: 270


In [58]:
print(img_paths_df['z_hsv_simclr'].values[0].shape)

(270, 256, 256)


In [60]:
# add hand-crafted features
def caculate_ndvi(X):
    # nir band is 0
    # red band is 1
    X = X.astype(np.float32)
    return (X[0] - X[1]) / (X[0] + X[1])

def caculate_ndwi(X):
    # nir band is 0
    # green band is 2
    X = X.astype(np.float32)
    return (X[2] - X[0]) / (X[2] + X[0])

def caclulate_gndvi(X):
    # nir band is 0
    # green band is 2
    X = X.astype(np.float32)
    return (X[0] - X[2]) / (X[0] + X[2])

img_paths_df['ndvi'] = img_paths_df['X_norm'].apply(caculate_ndvi)
img_paths_df['ndwi'] = img_paths_df['X_norm'].apply(caculate_ndwi)
img_paths_df['gndvi'] = img_paths_df['X_norm'].apply(caclulate_gndvi)

# append the hand-crafted features to the model output
for weights in ['imagenet', 'hsv_simclr']:
    deep_features = np.stack(img_paths_df[f'z_{weights}'].values)

    print(np.stack(img_paths_df['X'].values)[:,0][:, None].shape)
    print(np.stack(img_paths_df['ndvi'].values)[:, None].shape)

    # print(X.shape)
    aggregated_features = np.concatenate([
        deep_features, 
        np.stack(img_paths_df['ndvi'].values)[:, None], 
        np.stack(img_paths_df['ndwi'].values)[:, None], 
        np.stack(img_paths_df['gndvi'].values)[:, None], 
        np.stack(img_paths_df['X'].values)[:,0][:, None],
        np.stack(img_paths_df['X'].values)[:,1][:, None],
        np.stack(img_paths_df['X'].values)[:,2][:, None]],
    axis=1)
    img_paths_df[f'aggregated_features_{weights}'] = list(aggregated_features)

    n_features = aggregated_features.shape[1]
    print(f'Total number of features: {n_features}')

(90, 1, 256, 256)
(90, 1, 256, 256)
Total number of features: 276
(90, 1, 256, 256)
(90, 1, 256, 256)
Total number of features: 276


In [ ]:
train_df, test_df = train_test_split(img_paths_df, test_size=0.3, random_state=1701)

print(f'Train images: {len(train_df)}')
print(f'Test images: {len(test_df)}')

offset = (256 - 192) // 2 # only keep the center 192x192 pixels due to edge effects

y_preds = {}

for model in ['imagenet', 'hsv_simclr', 'none']:
    print("Evaluating model weights: ", model)
    
    if model == 'none':
        X_train = np.stack(train_df[f'aggregated_features_imagenet'])[:, :, offset:-offset, offset:-offset].transpose(0, 2, 3, 1).reshape(-1, n_features)[:, -6:]
        X_test = np.stack(test_df[f'aggregated_features_imagenet'])[:, :, offset:-offset, offset:-offset].transpose(0, 2, 3, 1).reshape(-1, n_features)[:, -6:]

    else:
        X_train = np.stack(train_df[f'aggregated_features_{model}'])[:, :, offset:-offset, offset:-offset].transpose(0, 2, 3, 1).reshape(-1, n_features)
        X_test = np.stack(test_df[f'aggregated_features_{model}'])[:, :, offset:-offset, offset:-offset].transpose(0, 2, 3, 1).reshape(-1, n_features)
    
    y_train = np.stack(train_df['y'].values)[:, :, offset:-offset, offset:-offset].flatten()
    y_test = np.stack(test_df['y'].values)[:, :, offset:-offset, offset:-offset].flatten()
    
    print('Removing 0s')
    
    X_train = X_train[y_train != 0]
    y_train = y_train[y_train != 0] - 1 # 0-based index
    
    X_test = X_test[y_test != 0]
    y_test = y_test[y_test != 0] - 1 # 0-based index
        
    print(f'Train size: {X_train.shape[0]}')
    print(f'Test size: {X_test.shape[0]}')
    
    print(f'Oversampling')
    
    smote = SMOTE(sampling_strategy='auto', random_state=1701, k_neighbors=NearestNeighbors(n_neighbors=5, n_jobs=-1))
    X_train, y_train = smote.fit_resample(X_train, y_train)
    
    print(f'Train size after oversampling: {X_train.shape[0]}')
    
    print('Training XGBoost model')
    
    xgb = XGBClassifier(n_estimators=20, n_jobs=-1, random_state=1701)
    xgb.fit(X_train, y_train)
    
    print('Evaluating model')
    y_pred = xgb.predict(X_test)
    print(classification_report(y_test, y_pred))
    y_preds[model] = y_pred


Train images: 63
Test images: 27
Evaluating model weights:  imagenet
Removing 0s
Train size: 2258706
Test size: 948274
Oversampling


In [ ]:
X_train_spectral = X_train[:, n_features-6:]
X_test_spectral = X_test[:, n_features-6:]

model_spectral = XGBClassifier(n_estimators=100, n_jobs=-1)

model_spectral.fit(X_train_spectral, y_train)

print(model_spectral.score(X_test_spectral, y_test))

y_preds_spectral = model_spectral.predict(X_test_spectral)

print(classification_report(y_test, y_preds_spectral))

0.595919570992427
              precision    recall  f1-score   support

           0       0.89      0.84      0.86      1737
           1       0.35      0.71      0.47      2509
           2       0.66      0.54      0.59      9040
           3       0.36      0.66      0.47     12388
           4       0.89      0.68      0.77    262540
           5       0.41      0.64      0.50     51132
           6       0.10      0.10      0.10     51561
           7       0.26      0.69      0.38     12629

    accuracy                           0.60    403536
   macro avg       0.49      0.61      0.52    403536
weighted avg       0.68      0.60      0.62    403536



In [ ]:
# visualize predictions from each model

import matplotlib.pyplot as plt

y_test_sample = y_test.reshape(-1, 
y_preds_sample = y_preds.reshape(-1, 192, 192)[0]
y_preds_spectral_sample = y_preds_spectral.reshape(-1, 192, 192)[0]

fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].imshow(y_test_sample)
axs[0].set_title('Ground Truth')

axs[1].imshow(y_preds_sample)
axs[1].set_title('Deep + Spectral + Hand-crafted Features')

axs[2].imshow(y_preds_spectral_sample)
axs[2].set_title('Spectral + Hand-Crafted Features Only')


ValueError: cannot reshape array of size 403536 into shape (64,64)

In [68]:
y_test_reshaped = y_test.reshape(11, 192, 192, 1)

ValueError: cannot reshape array of size 979053 into shape (11,192,192,1)

635.2448346897438